# Local RAG Experiment Template

로컬 Jupyter 환경에서 RAG config를 선택하고, ingest/retrieve/chat/evaluate 흐름을 확인하는 템플릿입니다.

## 1. 실험 기준

- 기본 config: `configs/experiments/rag/rag_smoke_test.yaml`
- 비교 config: `configs/experiments/rag/rag_smoke_hybrid.yaml`
- 평가 질문: `data/rag_smoke/eval_questions.csv`

In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd()
RAG_CONFIG = Path("configs/experiments/rag/rag_smoke_test.yaml")
COMPARE_CONFIG = Path("configs/experiments/rag/rag_smoke_hybrid.yaml")
QUESTION = "예산은 얼마야?"
OUTPUT_DIR = Path("experiments/rag_smoke_test")
EVAL_QUESTIONS = Path("data/rag_smoke/eval_questions.csv")

In [ ]:
def run(command: str) -> None:
    print(f"$ {command}")
    result = __import__("subprocess").run(command, shell=True, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"command failed: {command}")

def show_json(path: str | Path):
    path = Path(path)
    if not path.exists():
        print(f"missing: {path}")
        return {}
    return json.loads(path.read_text(encoding="utf-8"))

def show_csv(path: str | Path, n: int = 5):
    path = Path(path)
    if not path.exists():
        print(f"missing: {path}")
        return pd.DataFrame()
    return pd.read_csv(path).head(n)

def show_jsonl(path: str | Path, n: int = 3):
    path = Path(path)
    if not path.exists():
        print(f"missing: {path}")
        return []
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    return rows[:n]

## 2. 입력 확인

원본 문서와 평가 질문이 준비되어 있는지 확인합니다.

In [ ]:
print(RAG_CONFIG, RAG_CONFIG.exists())
print(EVAL_QUESTIONS, EVAL_QUESTIONS.exists())
show_csv(EVAL_QUESTIONS)

## 3. 실행 전 config 점검

In [ ]:
run(f"python scripts/check_rag_pipeline.py --config {RAG_CONFIG} --project-root .")

## 4. Ingest 실행

문서를 읽고 chunk와 embedding 산출물을 만듭니다.

In [ ]:
run(f"python scripts/run_rag_ingest.py --config {RAG_CONFIG} --project-root .")
show_csv(OUTPUT_DIR / "chunks.csv")

## 5. 검색 결과 확인

In [ ]:
run(f"python scripts/run_rag_retrieve.py --config {RAG_CONFIG} --project-root . --question '{QUESTION}'")

## 6. 답변과 평가 실행

In [ ]:
run(f"python scripts/run_rag_chat.py --config {RAG_CONFIG} --project-root . --question '{QUESTION}'")
run(f"python scripts/run_rag_chat.py --config {RAG_CONFIG} --project-root . --evaluate")

## 7. Metric과 실패 사례 확인

In [ ]:
show_json(OUTPUT_DIR / "metrics.json")

In [ ]:
display(show_csv(OUTPUT_DIR / "bad_retrievals.csv"))
display(show_csv(OUTPUT_DIR / "unsupported_answers.csv"))
display(show_csv(OUTPUT_DIR / "failed_questions.csv"))

## 8. 답변과 citation 확인

In [ ]:
show_jsonl(OUTPUT_DIR / "answers.jsonl")

## 9. Retriever 비교 리포트

In [ ]:
run("python scripts/compare_rag_retrievers.py --project-root .")
show_csv("reports/rag_retriever_comparison.csv")